In [1]:
import os
import time
import json
import pandas as pd
from urllib.parse import quote_plus

from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import (
    TimeoutException,
    NoSuchElementException,
    ElementClickInterceptedException,
    StaleElementReferenceException,
)

In [2]:
CONFIG = {
    "query": "برنامه نویس پایتون",

    "search_url": "https://jobinja.ir/jobs?filters[keywords][]={query}",
    "base_url": "https://jobinja.ir",

    "item_selector": "li.c-jobListView__item, div.c-jobListView__item, [class*='JobCard']",
    "title_selector": "h2.c-jobListView__jobTitle a, h2 a[href*='/job/'], a.c-jobListView__title",

    "company_selector": (
        "span.c-jobListView__companyName, a.c-jobListView__companyName, "
        "[class*='companyName'], [class*='company-name'], [class*='Company']"
    ),

    "location_selector": (
        "span.c-jobListView__location, [class*='location'], [class*='Location']"
    ),

    "next_selectors": [
        "a[rel='next']",
        "li.next a",
        "a.next",
        "a.pagination__next",
    ],

    "max_pages": 3,
    "wait_seconds": 20,
}

In [3]:
def create_driver():
    options = Options()

    profile_dir = os.path.expanduser("~/selenium-jobinja-profile")
    options.add_argument(f"--user-data-dir={profile_dir}")

    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")

    options.add_argument("--window-size=1920,1080")
    options.add_argument("--lang=fa")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/151.0.0.0 Safari/537.36"
    )


    driver = webdriver.Chrome(options=options)
    return driver


In [4]:
import re

def parse_current_page(html, page_number):
    soup = BeautifulSoup(html, "html.parser")
    records = []

    items = soup.select(CONFIG["item_selector"])
    if not items:
        items = soup.find_all(["li", "div"], class_=lambda x: x and "job" in str(x).lower())

    print(f"  → تعداد کارت‌های پیدا شده: {len(items)}")

    for item in items:
        title_tag = item.select_one(CONFIG["title_selector"])
        if not title_tag:
            title_tag = item.find(["h2", "h3", "a"], string=True)
        title = title_tag.get_text(strip=True) if title_tag else "نامشخص"

        company_tag = item.select_one(CONFIG["company_selector"])
        company = company_tag.get_text(strip=True) if company_tag else "نامشخص"
        if company == "نامشخص":
            for node in item.find_all(string=True):
                t = node.strip()
                if "|" in t and 3 < len(t) < 80:
                    company = t
                    break

        location_tag = item.select_one(CONFIG["location_selector"])
        location = location_tag.get_text(strip=True) if location_tag else "نامشخص"
        if location == "نامشخص":
            cities = ["تهران", "اصفهان", "مشهد", "شیراز", "تبریز", "کرج", "البرز",
                      "قم", "اهواز", "رشت", "کرمان", "یزد", "قزوین", "همدان",
                      "خراسان", "فارس", "مازندران", "گیلان", "آذربایجان"]
            for city in cities:
                if city in item.get_text():
                    location = city
                    break

        if title != "نامشخص":
            records.append({
                "صفحه": page_number,
                "عنوان شغل": title,
                "نام شرکت": company,
                "مکان": location,
            })

    return records


In [5]:
def find_next_button(driver):
    for selector in CONFIG["next_selectors"]:
        elements = driver.find_elements(By.CSS_SELECTOR, selector)
        for element in elements:
            try:
                if element.is_displayed():
                    return element
            except StaleElementReferenceException:
                continue

    fallback_xpaths = [
        "//a[contains(., 'بعدی') or contains(., 'Next')]",
        "//a[contains(@class, 'next')]",
        "//li[contains(@class, 'next')]/a",
    ]

    for xpath in fallback_xpaths:
        elements = driver.find_elements(By.XPATH, xpath)
        for element in elements:
            try:
                if element.is_displayed():
                    return element
            except StaleElementReferenceException:
                continue

    return None

In [6]:
def click_next_page(driver, wait, old_url):
    next_button = find_next_button(driver)
    if not next_button:
        return False

    old_items = driver.find_elements(By.CSS_SELECTOR, CONFIG["item_selector"])
    old_first_item = old_items[0] if old_items else None

    try:
        driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", next_button)
        time.sleep(0.5)

        try:
            next_button.click()
        except ElementClickInterceptedException:
            driver.execute_script("arguments[0].click();", next_button)

        try:
            wait.until(lambda d: d.current_url != old_url)
        except TimeoutException:
            pass

        if old_first_item is not None:
            try:
                wait.until(EC.staleness_of(old_first_item))
            except TimeoutException:
                pass

        wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "body")))
        time.sleep(2)
        return True

    except Exception:
        return False

In [7]:
def save_outputs(records):
    if not records:
        print("❌ هیچ رکوردی برای ذخیره وجود ندارد.")
        return None

    df = pd.DataFrame(records)
    df = df[["صفحه", "عنوان شغل", "نام شرکت", "مکان"]]

    csv_file = "jobinja_jobs.csv"
    json_file = "jobinja_jobs.json"

    df.to_csv(csv_file, index=False, encoding="utf-8-sig")

    with open(json_file, "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)

    print(f"\n✅ خروجی‌ها ذخیره شدند:")
    print(f"  📊 {csv_file}")
    print(f"  📄 {json_file}")
    print(f"\n📈 مجموع رکوردها: {len(records)}")
    print("\n📋 نمونه داده‌های استخراج‌شده:")
    print(df.head(10).to_string())

    return df

In [8]:
def main():
    driver = create_driver()
    wait = WebDriverWait(driver, CONFIG["wait_seconds"])
    all_records = []

    try:
        search_url = CONFIG["search_url"].format(query=quote_plus(CONFIG["query"]))
        print(f"🔍 در حال باز کردن صفحه جستجو:")
        print(f"   {search_url}\n")
        driver.get(search_url)

        time.sleep(4)

        for page_number in range(1, CONFIG["max_pages"] + 1):
            print(f"\n📄 در حال استخراج صفحه {page_number}...")

            try:
                wait.until(EC.presence_of_element_located((By.TAG_NAME, "body")))
                time.sleep(2)
            except TimeoutException:
                pass

            page_records = parse_current_page(driver.page_source, page_number)

            if not page_records:
                print(f"  ⚠️ هیچ رکوردی در صفحه {page_number} پیدا نشد.")
                break

            all_records.extend(page_records)
            print(f"  ✓ {len(page_records)} آگهی شغلی استخراج شد.")

            if page_number == CONFIG["max_pages"]:
                break

            current_url = driver.current_url
            if not click_next_page(driver, wait, current_url):
                print("  ⚠️ دکمه صفحه بعد پیدا نشد یا کلیک انجام نشد.")
                break

        if all_records:
            save_outputs(all_records)
        else:
            print("\n❌ هیچ داده‌ای استخراج نشد.")
            print("💡 راهنما: سایت جابینجا ممکن است ساختار HTML خود را تغییر داده باشد.")
            print("   روی یکی از آگهی‌ها راست‌کلیک کنید و Inspect را بزنید")
            print("   سپس سلکتورهای CONFIG را بر اساس کلاس‌های جدید به‌روزرسانی کنید.")

    finally:
        driver.quit()



In [9]:

if __name__ == "__main__":
    main()


🔍 در حال باز کردن صفحه جستجو:
   https://jobinja.ir/jobs?filters[keywords][]=%D8%A8%D8%B1%D9%86%D8%A7%D9%85%D9%87+%D9%86%D9%88%DB%8C%D8%B3+%D9%BE%D8%A7%DB%8C%D8%AA%D9%88%D9%86


📄 در حال استخراج صفحه 1...
  → تعداد کارت‌های پیدا شده: 20
  ✓ 20 آگهی شغلی استخراج شد.

📄 در حال استخراج صفحه 2...
  → تعداد کارت‌های پیدا شده: 20
  ✓ 20 آگهی شغلی استخراج شد.

📄 در حال استخراج صفحه 3...
  → تعداد کارت‌های پیدا شده: 20
  ✓ 20 آگهی شغلی استخراج شد.

✅ خروجی‌ها ذخیره شدند:
  📊 jobinja_jobs.csv
  📄 jobinja_jobs.json

📈 مجموع رکوردها: 60

📋 نمونه داده‌های استخراج‌شده:
   صفحه                                              عنوان شغل                                             نام شرکت    مکان
0     1                    برنامه‌نویس پایتون (Python-دورکاری)                استارت آپ پل دیزاینرز  | Poldesigners   تهران
1     1  توسعه دهنده پایتون (Python Software Engineer-AI Team)  رهند هوشمند نوین داده | Rahand Hoshmand Novin Dadeh   تهران
2     1              برنامه‌نویس پایتون (Python/Django-اصفهان)  د